# Standing Shoulder Internal/External Rotation - TCN

Este notebook usa `Standing_Shoulder_Internal_External_Rotation.py` para:
- Entrada: ventana temporal de frames desde JSON (`jsonEjemplo.json`)
- Features biomecanicas (angulo hombro-brazo, altura relativa, elevacion escapular, tronco, simetria, velocidad angular, rango maximo)
- Modelo: TCN
- Salidas: correcta/incorrecta, severidad y errores detectados
- Objetivo principal: maximizar sensibilidad/recall de ejecuciones incorrectas

## 0. Dependencias
Si usas Python 3.14, TensorFlow puede no estar disponible.
Para entrenar TCN, usa preferiblemente Python 3.11 o 3.12.

In [1]:
# Ejecuta solo si necesitas instalar en este kernel
%pip install -q numpy pandas scikit-learn joblib tensorflow

Note: you may need to restart the kernel to use updated packages.


## 1. Importaciones

In [2]:
from pathlib import Path
import json
import importlib
import numpy as np
import joblib

import Standing_Shoulder_Internal_External_Rotation as ssr
ssr = importlib.reload(ssr)

print(f'BASE_DIR: {Path.cwd()}')
print(f'JSON ejemplo: {ssr.JSON_EXAMPLE_PATH}')
print(f'Dataset real esperado: {ssr.DATASET_ROOT}')
print(f'Artifacts: {ssr.ARTIFACTS_DIR}')

BASE_DIR: c:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_internal_external_rotation
JSON ejemplo: C:\Users\marco\notebooks\modelo_rehabilitacion\jsonEjemplo.json
Dataset real esperado: C:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_internal_external_rotation\standing_shoulder_internal_external_rotation_json
Artifacts: C:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_internal_external_rotation\standing_shoulder_internal_external_rotation_artifacts


## 2. Vista rapida del JSON de entrada

In [3]:
payload = ssr.load_json_payload(ssr.JSON_EXAMPLE_PATH)

print('Claves de primer nivel:', list(payload.keys()))
if 'puntos_clave' in payload:
    print('Joints disponibles en ejemplo:', list(payload['puntos_clave'].keys()))
elif 'frames' in payload:
    print('Cantidad de frames en ejemplo:', len(payload['frames']))
else:
    print('Formato detectado:', type(payload))

Claves de primer nivel: ['metadatos', 'puntos_clave']
Joints disponibles en ejemplo: ['nariz', 'hombro_d', 'codo_d', 'muñeca_d']


## 3. Construccion de dataset

Primero usa tu dataset real UI-PRMD desde:

- `Segmented Movements/Segmented Movements/Kinect/Positions`

- `Incorrect Segmented Movements/Incorrect Segmented Movements/Kinect/Positions`

- `Movements/Movements/Kinect/Positions`

- `Incorrect Movements/Incorrect Movements/Kinect/Positions`



Si no encuentra secuencias m07 validas, recien ahi hace fallback a JSON y luego sintetico.

In [4]:
X, y, meta, dataset_source = ssr.get_dataset()
print(f'dataset_source: {dataset_source}')
print(f'X shape: {X.shape}')
print(f'y balance: {np.bincount(y)}')

coverage = meta.attrs.get('coverage', {})
if coverage:
    print('\nCobertura loader m07:')
    for k, v in coverage.items():
        print(f'  {k}: {v}')

if 'folder_split' in meta.columns:
    print('\nDistribucion por carpeta origen:')
    print(meta.groupby(['folder_split', 'label']).size().unstack(fill_value=0))

meta.head()

dataset_source: synthetic_bootstrap
X shape: (480, 48, 10)
y balance: [240 240]


,sequence_id,subject_id,label,source,active_errors,max_shoulder_angle_deg,mean_shoulder_angle_deg,trunk_tilt_p95_deg,shoulder_elevation_p95,symmetry_p95_deg,velocity_std_deg,wrist_above_elbow_mean,valid_points_ratio_mean
0,synthetic_0_000,1,0,synthetic_from_jsonEjemplo,,168.145569,153.023300,40.672478,0.106023,14.458708,2.237702,-0.488102,1.0
1,synthetic_0_001,2,0,synthetic_from_jsonEjemplo,,169.528137,153.103775,38.037167,0.044809,12.771025,1.956798,-0.486002,1.0
2,synthetic_0_002,3,0,synthetic_from_jsonEjemplo,,174.520233,154.433594,39.811344,0.111077,12.986032,3.411663,-0.487894,1.0
3,synthetic_0_003,4,0,synthetic_from_jsonEjemplo,,170.823639,157.342804,37.970367,0.082210,10.082978,2.810103,-0.485686,1.0
4,synthetic_0_004,5,0,synthetic_from_jsonEjemplo,,164.027710,154.948486,39.689640,0.097572,12.491031,2.811654,-0.489899,1.0


## 4. Entrenamiento del TCN (foco en recall)
Esta celda entrena, ajusta umbral para recall objetivo y exporta artifacts.

In [5]:
results = None
try:
    results = ssr.train_and_export()
    print('Entrenamiento completado.')
    print('Metricas principales:')
    for k, v in results['metrics'].items():
        print(f'  {k}: {v}')
except RuntimeError as exc:
    print(str(exc))
    print('Sugerencia: usa un entorno Python 3.11/3.12 para TensorFlow y vuelve a ejecutar.')

=== Standing Shoulder Internal/External Rotation / TCN ===
Dataset usado        : synthetic_bootstrap
Muestras totales     : 480
Threshold optimizado : 0.8809
Recall incorrecto    : 1.0000
Precision incorrecto : 1.0000
F1 incorrecto        : 1.0000
ROC-AUC              : 1.0000
Matriz de confusión:
[[60  0]
 [ 0 60]]
Reporte de clasificación:
              precision    recall  f1-score   support

    correcta     1.0000    1.0000    1.0000        60
  incorrecta     1.0000    1.0000    1.0000        60

    accuracy                         1.0000       120
   macro avg     1.0000    1.0000    1.0000       120
weighted avg     1.0000    1.0000    1.0000       120

Demo de inferencia:
{
  "exercise": "Standing Shoulder Internal/External Rotation",
  "movement_id": 7,
  "label": 1,
  "classification": "incorrecta",
  "probability_incorrect": 0.9924,
  "threshold": 0.8809,
  "severity": "severa",
  "errors_detected": [
    "codo_separado_del_torso",
    "inclinacion_excesiva_tronco",
    "

## 5. Inferencia demo
Salida esperada:
- etiqueta: correcta / incorrecta
- severidad
- errores_detectados

In [6]:
try:
    tf = ssr.require_tensorflow()
    bundle_path = ssr.ARTIFACTS_DIR / 'standing_shoulder_internal_external_rotation_bundle.joblib'
    model_path = ssr.ARTIFACTS_DIR / 'standing_shoulder_internal_external_rotation_tcn.keras'

    if not bundle_path.exists() or not model_path.exists():
        raise FileNotFoundError('No existen artifacts aun. Ejecuta la celda de entrenamiento primero.')

    bundle = joblib.load(bundle_path)
    model = tf.keras.models.load_model(model_path)

    demo_payload = ssr.load_json_payload(ssr.JSON_EXAMPLE_PATH)
    demo_frames, _ = ssr.generate_temporal_window_from_example(
        payload=demo_payload,
        rng=np.random.default_rng(123),
        incorrect=True,
        n_frames=ssr.WINDOW_SIZE,
    )

    pred = ssr.infer_from_window(demo_frames, model=model, bundle=bundle)
    print(json.dumps(pred, ensure_ascii=False, indent=2))
except Exception as exc:
    print(f'No se pudo ejecutar inferencia: {exc}')

{
  "exercise": "Standing Shoulder Internal/External Rotation",
  "movement_id": 7,
  "label": 1,
  "classification": "incorrecta",
  "probability_incorrect": 0.9886,
  "threshold": 0.8809,
  "severity": "severa",
  "errors_detected": [
    "codo_separado_del_torso",
    "inclinacion_excesiva_tronco",
    "elevacion_escapular_excesiva",
    "descenso_de_muneca_en_rotacion",
    "asimetria_derecha_izquierda",
    "velocidad_angular_irregular"
  ],
  "biomechanical_summary": {
    "max_shoulder_angle_deg": 178.0248,
    "mean_shoulder_angle_deg": 135.6553,
    "trunk_tilt_p95_deg": 47.8597,
    "shoulder_elevation_p95": 0.3547,
    "symmetry_p95_deg": 33.5665,
    "velocity_std_deg": 8.3392,
    "wrist_above_elbow_mean": -0.4738,
    "valid_points_ratio_mean": 1.0
  }
}


## 6. Uso en produccion (resumen)

1. Carga artifacts exportados (`.joblib` + `.keras`).
2. Recibe payload JSON con `frames` o `puntos_clave`.
3. Normaliza a secuencia con `ssr.coerce_frames(...)`.
4. Llama `ssr.infer_from_window(...)` y devuelve:
   - `etiqueta` (`correcta` / `incorrecta`)
   - `severidad`
   - `errores_detectados`
   - `probabilidad_incorrecta`
   - `resumen_biomecanico`

Ejemplo minimo de carga:

```python
from pathlib import Path
import joblib
import tensorflow as tf
import Standing_Shoulder_Internal_External_Rotation as ssr

artifacts = Path('standing_shoulder_internal_external_rotation_artifacts')
bundle = joblib.load(artifacts / 'standing_shoulder_internal_external_rotation_bundle.joblib')
model = tf.keras.models.load_model(artifacts / 'standing_shoulder_internal_external_rotation_tcn.keras')
```

Ejemplo minimo de prediccion:

```python
def predecir_rotacion(payload_json: dict) -> dict:
    frames = ssr.coerce_frames(payload_json)
    if len(frames) < 2:
        frames = frames * max(2, ssr.WINDOW_SIZE)
    return ssr.infer_from_window(frames_json=frames, model=model, bundle=bundle)
```

Nota: para entrenar TCN usa Python 3.11 o 3.12.

In [7]:
from pathlib import Path
import joblib
import tensorflow as tf
import Standing_Shoulder_Internal_External_Rotation as ssr

artifacts = Path('standing_shoulder_internal_external_rotation_artifacts')
bundle = joblib.load(artifacts / 'standing_shoulder_internal_external_rotation_bundle.joblib')
model = tf.keras.models.load_model(artifacts / 'standing_shoulder_internal_external_rotation_tcn.keras')

def predecir_rotacion(payload_json: dict) -> dict:
    frames = ssr.coerce_frames(payload_json)
    if len(frames) < 2:
        frames = frames * max(2, ssr.WINDOW_SIZE)
    return ssr.infer_from_window(frames_json=frames, model=model, bundle=bundle)




In [8]:
predecir_rotacion({ "metadatos": { "dispositivo": "Samsung Tab S3", "orientacion": "Portrait", "pantalla_px": {
"ancho": 1536, "alto": 2048 }, "timestamp_ms": 1710584200123 }, "puntos_clave": { "nariz":
{"x": 0.50, "y": 0.20, "z": -0.15}, "hombro_d": {"x": 0.40, "y": 0.35, "z": -0.05}, "codo_d": {"x":
0.35, "y": 0.50, "z": -0.08}, "muÃ±eca_d": {"x": 0.30, "y": 0.65, "z": -0.12} } })

{'exercise': 'Standing Shoulder Internal/External Rotation',
 'movement_id': 7,
 'label': 0,
 'classification': 'correcta',
 'probability_incorrect': 0.4971,
 'threshold': 0.8809,
 'severity': 'moderada',
 'errors_detected': [],
 'biomechanical_summary': {'max_shoulder_angle_deg': 164.7449,
  'mean_shoulder_angle_deg': 164.7449,
  'trunk_tilt_p95_deg': 33.6901,
  'shoulder_elevation_p95': 0.0,
  'symmetry_p95_deg': 0.0,
  'velocity_std_deg': 0.0,
  'wrist_above_elbow_mean': nan,
  'valid_points_ratio_mean': 0.5}}